# Model Tester

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [ ]:
import os
import sys

os.environ["KERAS_BACKEND"] = "tensorflow"
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "0"
# os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
# os.environ["XLA_FLAGS"] = (
#     "--xla_gpu_cuda_data_dir=/hpc/mp/apps/nvidia/hpc_sdk/24.5/Linux_x86_64/24.5/cuda"
# )

In [ ]:
import logging
import time

import healpy as hp
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
import keras
from keras.callbacks import (
    EarlyStopping,
    TerminateOnNaN,
    TensorBoard,
    ModelCheckpoint,
)
from keras.optimizers import Adam
from keras.optimizers.schedules import ExponentialDecay, LearningRateSchedule
from keras.metrics import RootMeanSquaredError

from mlpng import Core

# from mlpng.models import AutoModel
# from mlpng.models.modelcore import get_model_class
from mlpng.utils import (
    setup_logging,
    load_data,
    get_fisher,
    plot_predictions,
    plot_histogram,
    print_errors,
    plot_metrics,
    WarmupLearningRate,
    AttentionSchedule,
    try_init_wandb,
)
from mlpng.utils.dataloaders import HDF5Dataset
from deepsphere import HealpyGCNN, healpy_layers as hp_layer
from deepsphere.healpy_layers import HealpyChebyshev, HealpyPool

In [ ]:
print(f"TensorFlow version: {tf.__version__}")
print(f"CUDA version: {tf.sysconfig.get_build_info()['cuda_version']}")
print(f"cuDNN version: {tf.sysconfig.get_build_info()['cudnn_version']}")
print(f"keras version: {keras.__version__}")
print("Conda environment:", os.environ["CONDA_DEFAULT_ENV"])
print("Python executable:", sys.executable)
print("Python version:", sys.version)

In [ ]:
logger = setup_logging(__name__, level=logging.DEBUG)

In [ ]:
MAX_EPOCHS = 30
BATCH_SIZE = 64

args = [
    "settings/n128.json",
    "--nsims",
    "100",
    "--narray",
    "1000",
    "--pols",
    "T",
    "--fnl_range",
    "-100",
    "100",
]

core = Core(args)
indices = np.arange(hp.nside2npix(core.nside))

# fisher = get_fisher(core.file)
# fnl_scale = (core.fnl_max - core.fnl_min) / 2.0
# scaled_std = 1 / np.sqrt(fisher) / fnl_scale

In [ ]:
def to_tf(ds):
    return (
        tf.data.Dataset.from_generator(
            lambda: ds,
            output_signature=(
                tf.TensorSpec(shape=(len(indices), 1), dtype=tf.float32),
                tf.TensorSpec(shape=(), dtype=tf.float32),
            ),
        )
        .apply(tf.data.experimental.assert_cardinality(len(ds)))
        .cache()
        .shuffle(buffer_size=1024, reshuffle_each_iteration=True)
        .batch(
            BATCH_SIZE,
            drop_remainder=True,
            num_parallel_calls=tf.data.AUTOTUNE,
            deterministic=False,
        )
        .prefetch(tf.data.AUTOTUNE)
    )


f = 10
ds = HDF5Dataset(core.file, x_name="alm", y_name="fnl")
train_ds, test_ds, val_ds = ds.split(0.7 / f, 0.2 / f, 0.1 / f, verbose=True)
train_tf, test_tf, val_tf = map(to_tf, (train_ds, test_ds, val_ds))

In [ ]:
K = 11
depth = 1
cnn_depth = 2
F0 = 16

layers = []
for i in range(depth):
    for j in range(cnn_depth):
        layers.append(
            HealpyChebyshev(
                K=K,
                Fout=F0 * 2**i,
                use_bias=True,
                use_bn=False,
                activation="gelu" if j == 0 else None,
            )
        )
    layers.append(tf.keras.layers.LayerNormalization())
    layers.append(HealpyPool(p=1, pool_type="MAX"))

layers.append(keras.layers.Flatten())
layers.append(keras.layers.Dense(512, "gelu"))
layers.append(keras.layers.Dense(1))


decay_steps = len(train_ds) // BATCH_SIZE  # once per epoch
learning_rate = ExponentialDecay(1e-5, decay_steps, 0.95, staircase=True)

strategy = tf.distribute.MirroredStrategy()
with strategy.scope():
    indices = np.arange(hp.nside2npix(core.nside))
    gcnn = HealpyGCNN(
        nside=core.nside,
        indices=indices,
        layers=layers,
        n_neighbors=20,
        max_batch_size=BATCH_SIZE,
        initial_Fin=1,
    )
    gcnn.build(input_shape=(None, len(indices), 1))
    # gcnn.summary()

    # here is how to use the gcnn as a layer, KEEP this
    # inputs = keras.layers.Input(shape=(len(indices), 1))
    # layer = gcnn(inputs)
    # # layer = keras.layers.Flatten()(layer)
    # # layer = keras.layers.Dense(128, "relu")(layer)
    # # layer = keras.layers.Dense(32)(layer)
    # # layer = keras.layers.Dense(1)(layer)
    # model = keras.Model(inputs=inputs, outputs=layer)
    model = gcnn

    metrics = []  # RootMeanSquaredError(), "mae"]
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate), loss="mse", metrics=metrics
    )

model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    TensorBoard(log_dir=f"data/tensorboard/notebooks/{time.strftime('%Y%m%d-%H%M%S')}"),
    TerminateOnNaN(),
    keras.callbacks.ModelCheckpoint(
        f"{core.dirs['base']}/models/deepsphere_1.keras",
        monitor="val_loss",
        save_best_only=True,
        initial_value_threshold=400,
    ),
]

try_init_wandb(notes="model testing", tags=["notebook"], append_to=callbacks)

history = model.fit(
    train_tf, epochs=MAX_EPOCHS, validation_data=val_tf, callbacks=callbacks
)

In [ ]:
model.evaluate(test_tf, verbose=1)
preds = model.predict(test_tf, verbose=1)  # .flatten()
truth = np.concatenate([y for _, y in test_tf])

In [ ]:
fisher = get_fisher(core.file)
print_errors(truth, preds, fisher)

save_base = core.dirs["plot"]
plot_metrics(history, metrics=["loss"], save_file=save_base + "/ds-loss.png")
plot_predictions(
    truth, preds, fisher=fisher, show=True, save_file=save_base + "/ds-preds.png"
)
plot_histogram(truth, preds, show=True, save_file=save_base + "/ds-hist.png")